# 03 — Reverse Synthesis: GNN → Python Package

COGANT's reverse pipeline takes a GNN markdown file and synthesises a runnable Python package that mirrors the GNN topology: one dataclass field per hidden state, one `observe_*` function per observation modality, one `act_*` function per action, and so on.

In this notebook we:
1. Write a hand-crafted GNN string (a mini POMDP).
2. Parse it with `cogant.reverse.parser.parse_gnn`.
3. Plan the package layout with `cogant.reverse.planner.plan_package`.
4. Synthesise the package to disk with `cogant.reverse.synthesizer.synthesize_package`.
5. Inspect the generated `matrices.py` and assert it is valid Python.

Reverse synthesis is the foundation of the round-trip test in notebook 04.

## 1. A hand-written mini POMDP in GNN v1.1

This is deliberately tiny: 2 hidden states, 2 observations, 2 actions. It's the minimum useful POMDP and it makes the generated package easy to read.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

GNN_TEXT = """## GNNSection
mini_pomdp

## GNNVersionAndFlags
GNN v1

## ModelName
mini_pomdp

## ModelAnnotation
Hand-written 2-state 2-observation 2-action POMDP used by the COGANT tutorial notebooks.

## StateSpaceBlock
s_f0[2,1,type=int]
o_m0[2,1,type=int]
u_c0[2,1,type=int]
A_m0[2,2,type=float]
B_f0[2,2,2,type=float]
C_m0[2,1,type=float]
D_f0[2,1,type=float]

## Connections
(D_f0) > (s_f0)
(s_f0, B_f0) > (s_f0)
(s_f0, A_m0) > (o_m0)
(C_m0) > (o_m0)

## InitialParameterization
D_f0={ (0.5, 0.5) }
A_m0={ ((0.9, 0.1), (0.1, 0.9)) }
C_m0={ (1.0, 0.0) }
B_f0=identity(2,2,2)

## Time
Static

## ActInfOntologyAnnotation
s_f0=HiddenState
o_m0=Observation
u_c0=Action
A_m0=LikelihoodMatrix
B_f0=TransitionMatrix
C_m0=Preference
D_f0=PriorBelief
"""

print(f"GNN text length: {len(GNN_TEXT)} chars")

## 2. Parse with `parse_gnn`

`parse_gnn` accepts either a filesystem path or a raw string (it auto-detects). We write the text to a temp file to exercise the path code-path, which is what the CLI uses.

In [ ]:
from cogant.reverse.parser import parse_gnn

workdir = Path(tempfile.mkdtemp(prefix="cogant_nb03_"))
print(f"Workdir: {workdir}")

gnn_path = workdir / "mini_pomdp.gnn.md"
gnn_path.write_text(GNN_TEXT, encoding="utf-8")

model = parse_gnn(gnn_path)
print(f"Model name     : {model.model_name}")
print(f"Hidden states  : {model.hidden_states}")
print(f"Observations   : {model.observations}")
print(f"Actions        : {model.actions}")
print(f"Annotations    : {model.annotations}")
print(f"Cardinalities  : {model.cardinalities}")

## 3. Plan the package

`plan_package` is a pure function: it inspects the `ReverseGNNModel` and decides, for each semantic role, which Python construct to emit and under which module (`state.py`, `observe.py`, `act.py`, …).

In [ ]:
from cogant.reverse.planner import plan_package

plan = plan_package(model)
print(f"Package name      : {plan.package_name}")
print(f"State vars        : {[n.name for n in plan.state_vars]}")
print(f"Observe functions : {[n.name for n in plan.obs_functions]}")
print(f"Action methods    : {[n.name for n in plan.action_methods]}")
print(f"Policies          : {[n.name for n in plan.policy_functions]}")
print(f"Constraints       : {[n.name for n in plan.constraint_checks]}")

## 4. Synthesize the package

`synthesize_package` writes the package directory under `output_dir`. Inside you'll find `__init__.py`, `state.py`, `observe.py`, `act.py`, `policy.py`, `constraints.py`, `matrices.py`, `main.py`, and a `tests/test_smoke.py`.

In [ ]:
from cogant.reverse.synthesizer import synthesize_package

output_dir = workdir / "synthesized"
output_dir.mkdir(exist_ok=True)

package_path = synthesize_package(plan, model, output_dir)
print(f"Package written to: {package_path}")
print("Files:")
for p in sorted(package_path.rglob("*.py")):
    rel = p.relative_to(package_path)
    print(f"  {rel}  ({p.stat().st_size} bytes)")

## 5. Inspect `matrices.py`

`matrices.py` contains the A/B/C/D numpy arrays as source-level constants. It's the cleanest file to eyeball because it maps 1-to-1 back to the `## InitialParameterization` section of the GNN.

In [ ]:
matrices_path = package_path / "matrices.py"
matrices_src = matrices_path.read_text(encoding="utf-8")
print(matrices_src)

## 6. Assertions — the file exists and is valid Python

We compile the source via `compile(..., mode="exec")` — no import side-effects, just a syntax check.

In [ ]:
import ast

assert matrices_path.exists(), f"{matrices_path} does not exist"
assert matrices_path.stat().st_size > 0, "matrices.py is empty"

# AST parse confirms it is syntactically valid Python.
tree = ast.parse(matrices_src, filename=str(matrices_path))
top_level_names = [n.targets[0].id for n in tree.body if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name)]
print(f"OK — matrices.py parses cleanly. Top-level assignments: {top_level_names}")